# 14 - Deploy Runtime
Deploy agents to AgentCore Runtime via CLI/SDK, invoke deployed agents, and check runtime status.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import time
import subprocess
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]
ACCOUNT_ID = config["ACCOUNT_ID"]

tracker = ResourceTracker()
agentcore = boto3.client("bedrock-agentcore", region_name=REGION)

print(f"\u2713 Config loaded (region={REGION}, prefix={PREFIX})")

## Step 2: Verify Agent Project Structure

Check that the agent directory has the required files for deployment.

In [ ]:
import os

AGENTS_DIR = Path("../agents")
AGENT_NAME = f"{PREFIX}-search"
AGENT_DIR = AGENTS_DIR / "search"

required_files = ["main.py", "Dockerfile", "pyproject.toml"]

print(f"Checking agent project: {AGENT_DIR}")
if AGENT_DIR.exists():
    for f in required_files:
        path = AGENT_DIR / f
        status = "\u2713" if path.exists() else "\u2717"
        print(f"  {status} {f}")
else:
    print(f"  \u231b Agent directory not found at {AGENT_DIR}")
    print(f"  Deployment will use SDK-based registration instead.")

## Step 3: Register Agent with AgentCore Runtime

Register the agent definition (name, config, resources) before deployment.

In [ ]:
# Check if agent already registered
existing_agents = agentcore.list_agents()
existing = next(
    (a for a in existing_agents.get("agents", []) if a["name"] == AGENT_NAME),
    None
)

if existing:
    agent_id = existing["id"]
    print(f"\u2713 Agent already registered: {agent_id}")
else:
    print(f"\u231b Registering agent: {AGENT_NAME}...")
    response = agentcore.create_agent(
        name=AGENT_NAME,
        description="OrgIQ Search agent - finds experts and documents in the knowledge graph",
        agentConfiguration={
            "runtime": {
                "containerImage": f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{AGENT_NAME}:latest",
                "port": 8080,
                "healthCheckPath": "/ping",
                "environment": {
                    "AURORA_CLUSTER_ARN": config["AURORA_CLUSTER_ARN"],
                    "AURORA_SECRET_ARN": config["AURORA_SECRET_ARN"],
                    "KB_ID": config["KB_ID"],
                    "REGION": REGION,
                    "MODEL_ID": config["MODEL_ID"]
                },
                "resources": {
                    "memory": 512,
                    "cpu": 256
                }
            }
        },
        tags={"project": PREFIX}
    )
    agent_id = response["id"]
    tracker.add_agentcore_agent(agent_id, AGENT_NAME)
    print(f"\u2713 Agent registered: {agent_id}")

## Step 4: Deploy via AgentCore CLI

The `agentcore deploy` command builds ARM64 Docker image via CodeBuild and deploys to Runtime.

In [ ]:
# Deploy using AgentCore CLI
# Note: This requires the agent directory with Dockerfile
if AGENT_DIR.exists():
    print(f"\u231b Deploying {AGENT_NAME} via AgentCore CLI...")
    print(f"  Source: {AGENT_DIR}")
    print(f"  This triggers remote CodeBuild (ARM64 image build).")
    print(f"  No local Docker required.\n")
    
    result = subprocess.run(
        ["agentcore", "deploy", "--name", AGENT_NAME],
        capture_output=True, text=True, cwd=str(AGENT_DIR),
        timeout=300
    )
    
    if result.returncode == 0:
        print(f"\u2713 Deployment initiated")
        print(result.stdout)
    else:
        print(f"\u231b Deployment output: {result.stderr[:200]}")
        print("  (This is expected if agent source is not yet written)")
else:
    print("\u231b Agent directory not found - skipping CLI deploy")
    print(f"  To deploy manually:")
    print(f"    cd agents/search && agentcore deploy")

## Step 5: Check Deployment Status

In [ ]:
# Check agent status via SDK
try:
    agent_info = agentcore.get_agent(agentId=agent_id)
    
    print(f"Agent: {agent_info['name']}")
    print(f"  ID: {agent_info['id']}")
    print(f"  Status: {agent_info.get('status', 'UNKNOWN')}")
    print(f"  Endpoint: {agent_info.get('endpoint', 'N/A')}")
    print(f"  Last deployed: {agent_info.get('lastDeployedAt', 'N/A')}")
    
    if agent_info.get('status') == 'ACTIVE':
        print(f"\n\u2713 Agent is ACTIVE and ready for invocation")
    else:
        print(f"\n\u231b Agent is not yet active (status: {agent_info.get('status')})")
except Exception as e:
    print(f"\u231b Unable to get agent status: {e}")

## Step 6: Check Status via CLI

In [ ]:
# Check status via AgentCore CLI
result = subprocess.run(
    ["agentcore", "status"],
    capture_output=True, text=True, timeout=30
)

if result.returncode == 0:
    print("AgentCore Status:")
    print(result.stdout)
else:
    print(f"\u231b CLI status check: {result.stderr[:100]}")

## Step 7: Invoke Deployed Agent via SDK

In [ ]:
# Invoke the deployed agent
try:
    invoke_response = agentcore.invoke_agent(
        agentId=agent_id,
        input={
            "messages": [
                {"role": "user", "content": "Who are the top Kubernetes experts?"}
            ]
        },
        sessionId="test-session-001"
    )
    
    output = invoke_response.get("output", {})
    print("Agent invocation response:")
    if "messages" in output:
        for msg in output["messages"]:
            print(f"  [{msg['role']}] {msg['content'][:200]}")
    else:
        print(f"  {json.dumps(output, indent=2)[:300]}")
    
    print("\n\u2713 Agent invocation successful")
except Exception as e:
    print(f"\u231b Agent invocation not available: {type(e).__name__}")
    print(f"  Agent may not be deployed yet. Deploy with:")
    print(f"    cd agents/search && agentcore deploy")

## Step 8: Invoke via CLI

In [ ]:
# Invoke via AgentCore CLI
result = subprocess.run(
    ["agentcore", "invoke", "--agent", AGENT_NAME, "--prompt", "Find authentication experts"],
    capture_output=True, text=True, timeout=60
)

if result.returncode == 0:
    print("CLI Invocation result:")
    print(result.stdout[:500])
    print("\n\u2713 CLI invocation successful")
else:
    print(f"\u231b CLI invocation: {result.stderr[:200]}")
    print("  (Expected if agent not yet deployed)")

## Step 9: View Agent Logs

In [ ]:
# Fetch recent agent logs
try:
    logs = agentcore.get_agent_logs(
        agentId=agent_id,
        limit=10
    )
    
    print(f"Recent logs for {AGENT_NAME}:")
    for entry in logs.get("logEntries", []):
        timestamp = entry.get("timestamp", "")
        message = entry.get("message", "")[:100]
        print(f"  [{timestamp}] {message}")
    
    print(f"\n\u2713 Logs retrieved")
except Exception as e:
    print(f"\u231b Logs not available: {type(e).__name__}")
    print(f"  Check CloudWatch: /aws/agentcore/{AGENT_NAME}")

## Step 10: Runtime Configuration Summary

In [ ]:
# Dockerfile requirements recap
print("AgentCore Runtime Requirements:")
print("  - Multistage: uv builder -> python:3.12-slim production")
print("  - Non-root user: bedrock_agentcore (UID 1000)")
print("  - Ports: 8080 (HTTP/AG-UI), 8000 (health/MCP)")
print("  - Health check: GET /ping")
print("  - CMD: opentelemetry-instrument python -m main")
print("  - Max image size: 1 GB")
print("  - Architecture: ARM64 (CodeBuild handles this)")
print()
print("Deployment flow:")
print("  1. agentcore deploy (from agent dir)")
print("  2. CodeBuild builds ARM64 Docker image")
print("  3. Image pushed to ECR")
print("  4. Runtime provisions microVM")
print("  5. Agent becomes ACTIVE")

## Step 11: Save Configuration

In [ ]:
agent_arns = config.get("AGENT_RUNTIME_ARNS", {})
agent_arns[AGENT_NAME] = f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:agent/{agent_id}"
config["AGENT_RUNTIME_ARNS"] = agent_arns

Path("config.json").write_text(json.dumps(config, indent=2))
print(f"\u2713 Config saved")
print(f"  AGENT_RUNTIME_ARNS[{AGENT_NAME}]: {agent_arns[AGENT_NAME]}")

## Summary

- [x] Agent project structure verified
- [x] Agent registered with AgentCore Runtime
- [x] Deployment via CLI demonstrated
- [x] Status checking (SDK and CLI)
- [x] Agent invocation via SDK
- [x] Agent invocation via CLI
- [x] Log retrieval
- [x] Runtime requirements documented

**Next:** [15-evaluations.ipynb](15-evaluations.ipynb) — Evaluate agent quality with built-in and custom evaluators